In [ ]:
#  scheduling part --  test version - it worked
import gurobipy as gp
import random
from gurobipy import GRB

# ========= 1. 基本数据 =========
N = 3                  # 管线数量
J = 20                # 真实事件点数量
T = 100.0              # 规划期长度

# 集合
N_set = range(N)
J_all = range(J + 2)       # k = 0,...,J+1 （含虚拟）
J_real = range(1, J + 1)   # 真实事件 1,...,J
J_interval = range(J + 1)  # 区间 k=0,...,J

# 参数
b_GR = 3.0
b_RE = 5.0
C_GR = 10.0
C_RE = 100.0
C_Leak = 10.0
M = 3.5                 # 大M

random.seed(42)
P_init = [random.uniform(3.21, 3.5) for _ in range(N)]
P_max = [3.5] * N
P_min = [3.0] * N
alpha_init = [random.uniform(0.005, 0.008) for _ in range(N)]
alpha_new = [alpha_init[i]/3 for i in range(N)]
V = [random.uniform(1, 3) for _ in range(N)]
K = [10**2 * 146 * V[i] / (8.314 * 293.15) for i in range(N)]
L_total = 4

# ========= 2. 创建模型 =========
m = gp.Model("event_based_maintenance")
m.Params.MIPGap = 0.001

# ========= 3. 决策变量 =========
x_GR = m.addVars(N_set, J_all, vtype=GRB.BINARY, name="x_GR")
x_RE = m.addVars(N_set, J_all, vtype=GRB.BINARY, name="x_RE")
z = m.addVars(N_set, J_all, vtype=GRB.BINARY, name="z")  # 是否已RE
delta_t = m.addVars(N_set, J_interval, vtype=GRB.CONTINUOUS, lb=0.0, name="delta_t")
t = m.addVars(N_set, J_all, vtype=GRB.INTEGER, lb=0.0, name="t")
t_end = m.addVars(N_set, J_all, vtype=GRB.INTEGER, lb=0.0, name="t_end")
P = m.addVars(N_set, J_all, vtype=GRB.CONTINUOUS, name="P")
P_end = m.addVars(N_set, J_all, vtype=GRB.CONTINUOUS, name="P_end")
Leak = m.addVars(N_set, J_interval, vtype=GRB.CONTINUOUS, lb=0.0, name="Leak")

# ========= 4. 目标函数 =========
m.setObjective(
    gp.quicksum(C_GR * x_GR[i, k] + C_RE * x_RE[i, k] for i in N_set for k in J_real) +
    gp.quicksum(C_Leak * Leak[i, k] for i in N_set for k in J_interval), 
    GRB.MINIMIZE
)

# ========= 5. 约束 =========

# 5.1 虚拟事件约束
for i in N_set:
    m.addConstr(t[i, 0] == 0.0)
    m.addConstr(t[i, J + 1] == T)
    m.addConstr(x_GR[i, 0] == 0)
    m.addConstr(x_RE[i, 0] == 0)
    m.addConstr(x_GR[i, J + 1] == 0)
    m.addConstr(x_RE[i, J + 1] == 0)
    m.addConstr(t_end[i, 0] == t[i, 0])
    m.addConstr(t_end[i, J + 1] == t[i, J + 1])

# 5.2 时间约束：严格递增 + 覆盖整个规划期
# epsilon = 0.005 * T  # 每个区间至少0.5%
epsilon = 1  # 每个区间至少1个时间单位
for i in N_set:
    m.addConstr(t[i, J] >= 0.95 * T, name=f"time_coverage_{i}")  # 确保前J个点覆盖95%
    for k in range(J + 1):
        m.addConstr(t[i, k + 1] >= t[i, k] + epsilon, name=f"time_min_{i}_{k}")
    
    # delta_t 定义
    m.addConstr(delta_t[i, 0] == t[i, 1] - t[i, 0])
    for k in range(1, J):
        m.addConstr(delta_t[i, k] == t[i, k + 1] - t[i, k])
    m.addConstr(delta_t[i, J] == t[i, J + 1] - t[i, J])
    
    # 不重叠约束
    for k in range(J):
        m.addConstr(t[i, k + 1] >= t_end[i, k])

# 5.3 维护结束时间
for i in N_set:
    for k in J_real:
        m.addConstr(t_end[i, k] == t[i, k] + b_GR * x_GR[i, k] + b_RE * x_RE[i, k])

# 5.4 每个事件最多一个维护
for i in N_set:
    for k in J_real:
        m.addConstr(x_GR[i, k] + x_RE[i, k] <= 1)

# 5.5 总维护次数
m.addConstr(gp.quicksum(x_GR[i, k] + x_RE[i, k] for i in N_set for k in J_real) <= L_total)

# 5.6 z变量：累计RE状态
for i in N_set:
    m.addConstr(z[i, 1] >= x_RE[i, 1])
    for k in range(2, J + 1):
        m.addConstr(z[i, k] >= z[i, k - 1])
        m.addConstr(z[i, k] >= x_RE[i, k])
        m.addConstr(z[i, k] <= z[i, k - 1] + x_RE[i, k])

# 5.7 P_end逻辑：维护时=P_max，不维护时=P当前
for i in N_set:
    m.addConstr(P_end[i, 0] == P_init[i])
    for k in J_real:
        m.addConstr(P_end[i, k] == P_max[i] * (x_GR[i, k] + x_RE[i, k]) + 
                           P[i, k] * (1 - x_GR[i, k] - x_RE[i, k]))

# 5.8 压力演化
for i in N_set:
    m.addConstr(P[i, 0] == P_init[i])
    m.addConstr(P[i, 1] == P_init[i] - alpha_init[i] * delta_t[i, 0])
    for k in range(1, J):
        m.addConstr(P[i, k + 1] == P_end[i, k] - 
                   (alpha_init[i] * (1 - z[i, k]) + alpha_new[i] * z[i, k]) * delta_t[i, k])

# 5.9 压力范围
for i in N_set:
    for k in J_all:
        m.addConstr(P[i, k] >= P_min[i])
        m.addConstr(P[i, k] <= P_max[i])

# 5.10 泄漏计算：每个区间的实际压力下降
for i in N_set:
    m.addConstr(Leak[i, 0] == (P_init[i] - P[i, 1]) * K[i])
    for k in range(1, J + 1):
        m.addConstr(Leak[i, k] == (P[i, k] - P[i, k + 1]) * K[i])

# ========= 6. 求解 =========
m.optimize()

# ========= 7. 输出结果 =========
if m.status == GRB.OPTIMAL:
    print(f"Optimal objective: {m.objVal:.4f}")
    
    for i in N_set:
        print(f"\n==== Caisson {i} maintenance schedule ====")
        tasks_i = []
        for k in J_real:
            if x_GR[i, k].X > 0.5:
                tasks_i.append((t[i, k].X, k, 'GR'))
            if x_RE[i, k].X > 0.5:
                tasks_i.append((t[i, k].X, k, 'RE'))
        
        tasks_i.sort(key=lambda x: x[0])
        if not tasks_i:
            print("  No maintenance tasks scheduled.")
        else:
            for time_k, k, typ in tasks_i:
                print(f"  Task at event k={k:2d}, time={time_k:6.2f}, type={typ}")
        
        # 显示关键压力轨迹
        print(f"  Key pressure points:")
        print(f"    t=0: P={P[i,0].X:.3f}")
        print(f"    t={t[i,J].X:.1f}: P={P[i,J].X:.3f}")
        print(f"    t={T}: P={P[i,J+1].X:.3f}")
else:
    print(f"Status: {m.status}")


Set parameter MIPGap to value 0.001
Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (win64)

CPU model: Intel(R) Core(TM) i5-10210U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 712 rows, 588 columns and 1683 nonzeros
Model fingerprint: 0xbe95dd97
Model has 117 quadratic constraints
Variable types: 258 continuous, 330 integer (198 binary)
Coefficient statistics:
  Matrix range     [6e-03, 2e+01]
  QMatrix range    [4e-03, 1e+00]
  QLMatrix range   [6e-03, 4e+00]
  Objective range  [1e+01, 1e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+02]
Presolve removed 138 rows and 228 columns
Presolve time: 0.04s
Presolved: 994 rows, 480 columns, 2985 nonzeros
Variable types: 240 continuous, 240 integer (177 binary)

Root relaxation: objective 1.665406e+01, 542 iterations, 0.02 seconds (0.01 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work

In [1]:
#  scheduling part --  test version - better results display
import gurobipy as gp
import random
from gurobipy import GRB

# ========= 1. Parameters =========
N = 10                 # Number of caissons
T = 100.0              # Planning horizon

# Other parameters
b_GR = 3.0                 # GR task duration
b_RE = 5.0                 # RE task duration
C_GR = 10.0                # Unit cost of GR task
C_RE = 100.0               # Unit cost of RE task
C_Leak = 10.0              # Unit cost of leakage per kg SF6
M = 3.5                    # big M
r_GR = 3                   # GR task requires 3 workers
r_RE = 5                   # RE task requires 5 workers
nMAX_GR_simultaneous = 8   # Maximum number of simultaneous GR tasks
nMAX_RE_simultaneous = 5   # Maximum number of simultaneous RE tasks
L_total = 20               # Total maintenance actions allowed

# Problem inputs
random.seed(42)
P_init = [random.uniform(3.21, 3.5) for _ in range(N)]
P_max = [3.5] * N
P_min = [3.01] * N
# alpha_init = [random.uniform(0.005, 0.008) for _ in range(N)]
alpha_init = [random.uniform(0.01, 0.08) for _ in range(N)]
alpha_new = [alpha_init[i]/3 for i in range(N)]
V = [random.uniform(1, 3) for _ in range(N)]
K = [10**2 * 146 * V[i] / (8.314 * 293.15) for i in range(N)]   # coefficient to translate pressure to leakage mass

# Determine J (number of real event points) based on critical time to reach P_min
T_crit = max((P_init[i]-P_min[i])/alpha_init[i] for i in range(N))
# J_min = max(12, int(T / 15))
# J_max = min(60, int(T / 2.5))
# J = max(J_min, min(J_max, int(T_crit * 1.3 / 2.5)))
J = round(T/T_crit) + 2

# Sets
N_set = range(N)
J_all = range(J + 2)       # k = 0,...,J+1 (including virtual)
J_real = range(1, J + 1)   # Real events 1,...,J
J_interval = range(J + 1)  # Intervals k=0,...,J

# ========= 2. Create model =========
m = gp.Model("event_based_maintenance")
m.Params.MIPGap = 0.001

# ========= 3. Decision variables =========
x_GR = m.addVars(N_set, J_all, vtype=GRB.BINARY, name="x_GR")
x_RE = m.addVars(N_set, J_all, vtype=GRB.BINARY, name="x_RE")
z = m.addVars(N_set, J_all, vtype=GRB.BINARY, name="z")  # Whether RE has been done
delta_t = m.addVars(N_set, J_interval, vtype=GRB.CONTINUOUS, lb=0.0, name="delta_t")
t = m.addVars(N_set, J_all, vtype=GRB.INTEGER, lb=0.0, name="t")
t_end = m.addVars(N_set, J_all, vtype=GRB.INTEGER, lb=0.0, name="t_end")
P = m.addVars(N_set, J_all, vtype=GRB.CONTINUOUS, name="P")
P_end = m.addVars(N_set, J_all, vtype=GRB.CONTINUOUS, name="P_end")
Leak = m.addVars(N_set, J_interval, vtype=GRB.CONTINUOUS, lb=0.0, name="Leak")

# ========= 4. Objective function =========
m.setObjective(
    gp.quicksum(C_GR * x_GR[i, k] + C_RE * x_RE[i, k] for i in N_set for k in J_real) +
    gp.quicksum(C_Leak * Leak[i, k] for i in N_set for k in J_interval), 
    GRB.MINIMIZE)

# ========= 5. Constraints =========

# 5.1 Virtual event constraints
for i in N_set:
    m.addConstr(t[i, 0] == 0.0)
    m.addConstr(t[i, J + 1] == T)
    m.addConstr(x_GR[i, 0] == 0)
    m.addConstr(x_RE[i, 0] == 0)
    m.addConstr(x_GR[i, J + 1] == 0)
    m.addConstr(x_RE[i, J + 1] == 0)
    m.addConstr(t_end[i, 0] == t[i, 0])
    m.addConstr(t_end[i, J + 1] == t[i, J + 1])

# 5.2 Time constraints: strictly increasing + covering the entire planning horizon
# epsilon = 0.005 * T  # Each interval at least 0.5%
epsilon = 0.1  # Each interval at least 0.1 time units
for i in N_set:
    m.addConstr(t[i, J] >= 0.80 * T, name=f"time_coverage_{i}")  # Ensure first J points cover 80% of T
    for k in range(J + 1):
        m.addConstr(t[i, k + 1] >= t[i, k] + epsilon, name=f"time_min_{i}_{k}")
    
    # delta_t definition
    m.addConstr(delta_t[i, 0] == t[i, 1] - t[i, 0])
    for k in range(1, J):
        m.addConstr(delta_t[i, k] == t[i, k + 1] - t[i, k])
    m.addConstr(delta_t[i, J] == t[i, J + 1] - t[i, J])
    
    # Non-overlapping constraints
    for k in range(J):
        m.addConstr(t[i, k + 1] >= t_end[i, k])

# 5.3 Maintenance end time
for i in N_set:
    for k in J_real:
        m.addConstr(t_end[i, k] == t[i, k] + b_GR * x_GR[i, k] + b_RE * x_RE[i, k])

# 5.4 Each event at most one maintenance
for i in N_set:
    for k in J_real:
        m.addConstr(x_GR[i, k] + x_RE[i, k] <= 1)

# 5.5 Total maintenance times
m.addConstr(gp.quicksum(x_GR[i, k] + x_RE[i, k] for i in N_set for k in J_real) <= L_total)

# 5.6 z variable: cumulative RE status
for i in N_set:
    m.addConstr(z[i, 1] >= x_RE[i, 1])
    for k in range(2, J + 1):
        m.addConstr(z[i, k] >= z[i, k - 1])
        m.addConstr(z[i, k] >= x_RE[i, k])
        m.addConstr(z[i, k] <= z[i, k - 1] + x_RE[i, k])

# 5.7 P_end logic: P_max if maintenance, P_current if no maintenance
for i in N_set:
    m.addConstr(P_end[i, 0] == P_init[i])
    for k in J_real:
        m.addConstr(P_end[i, k] == P_max[i] * (x_GR[i, k] + x_RE[i, k]) + 
                           P[i, k] * (1 - x_GR[i, k] - x_RE[i, k]))

# 5.8 Pressure evolution
for i in N_set:
    m.addConstr(P[i, 0] == P_init[i])
    m.addConstr(P[i, 1] == P_init[i] - alpha_init[i] * delta_t[i, 0])
    for k in range(1, J):
        m.addConstr(P[i, k + 1] == P_end[i, k] - (alpha_init[i] * (1 - z[i, k]) + alpha_new[i] * z[i, k]) * delta_t[i, k])

# 5.9 Pressure range
for i in N_set:
    for k in J_all:
        m.addConstr(P[i, k] >= P_min[i])
        m.addConstr(P[i, k] <= P_max[i])

# 5.10 Leakage calculation: actual pressure drop in each interval
for i in N_set:
    m.addConstr(Leak[i, 0] == (P_init[i] - P[i, 1]) * K[i])
    for k in range(1, J + 1):
        m.addConstr(Leak[i, k] == (P[i, k] - P[i, k + 1]) * K[i])

# 5.11 Prevent too many tasks starting at the same time (personnel-related constraints)
for k in J_real:
    m.addConstr(gp.quicksum(x_GR[i, k] for i in N_set) <= nMAX_GR_simultaneous)  # Max 8 GR
    m.addConstr(gp.quicksum(x_RE[i, k] for i in N_set) <= nMAX_RE_simultaneous)  # Max 5 RE

# ========= 6. Optimization =========
m.optimize()

# ========= 7. Output results =========
if m.status == GRB.OPTIMAL:
    print(f"Optimal objective: {m.objVal:.4f}")
    
    # 👷 Global capacity summary
    print(f"\n👷 PERSONNEL SUMMARY:")
    print(f"  Maximum number of simultaneous GR tasks: {nMAX_GR_simultaneous}")
    print(f"  Maximum number of simultaneous RE tasks: {nMAX_RE_simultaneous}")
    print()
    
    for i in N_set:
        print(f"\n{'='*70}")
        print(f"CAISSON {i} - P_init={P_init[i]:.3f}, α_init={alpha_init[i]:.4f}")
        print(f"{'='*70}")
        
        # 1. Maintenance schedule
        print("📋 MAINTENANCE SCHEDULE:")
        tasks_i = []
        for k in J_real:
            if x_GR[i, k].X > 0.5:
                tasks_i.append((t[i, k].X, k, 'GR', P[i,k].X, P_end[i,k].X, r_GR))  # GR requires 3 workers
            if x_RE[i, k].X > 0.5:
                tasks_i.append((t[i, k].X, k, 'RE', P[i,k].X, P_end[i,k].X, r_RE))  # RE requires 5 workers
        tasks_i.sort(key=lambda x: x[0])
        
        if not tasks_i:
            print("  ❌ No maintenance tasks scheduled")
        else:
            print(f"  {'k':>3} | {'t':>7} | {'Type':>3} | {'Workers':>8} | {'P_start':>8} → {'P_end':>8}")
            print(f"  {'-'*60}")
            for time_k, k, typ, P_start, P_end_val, workers in tasks_i:
                print(f"  {k:3d} | {time_k:7.1f} | {typ:>3} | {workers:6d} | "
                      f"{P_start:7.3f} → {P_end_val:7.3f}")
        
        print()
        
        # 2. Detailed decisions for each potential task point
        print("📊 ALL POTENTIAL TASK POINTS:")
        show_points = list(range(1, 11)) + list(range(J-4, J+1))
        for _, k, _, _, _, _ in tasks_i:
            if k not in show_points:
                show_points.append(k)
        show_points = sorted(set(show_points))
        
        print(f"  {'k':>3} | {'t':>7} | {'δt':>5} | {'P':>6} | {'Type':>6} | {'Workers':>8} | {'P_end':>8} | {'ΔP':>7}")
        print(f"  {'-'*80}")
        for k in show_points:
            if k in J_real:
                if x_GR[i,k].X > 0.5:
                    maint_type, workers = 'GR', 3
                elif x_RE[i,k].X > 0.5:
                    maint_type, workers = 'RE', 5
                else:
                    maint_type, workers = 'NONE', 0
                
                deltaP = P_end[i,k].X - P[i,k].X
                print(f"  {k:3d} | {t[i,k].X:7.1f} | {delta_t[i,k-1].X:5.2f} | "
                      f"{P[i,k].X:6.3f} | {maint_type:>6} | {workers:6d} | "
                      f"{P_end[i,k].X:8.3f} | {deltaP:+7.3f}")
        
        # 3. Pressure trajectory summary
        print()
        print("📈 PRESSURE TRAJECTORY SUMMARY:")
        print(f"  t=0.0:   P={P[i,0].X:5.3f}")
        print(f"  t={t[i,J//4].X:5.1f}: P={P[i,J//4].X:5.3f}")
        print(f"  t={t[i,J//2].X:5.1f}: P={P[i,J//2].X:5.3f}")
        print(f"  t={t[i,3*J//4].X:5.1f}: P={P[i,3*J//4].X:5.3f}")
        print(f"  t={t[i,J].X:5.1f}: P={P[i,J].X:5.3f}")
        print(f"  t={T:5.1f}: P={P[i,J+1].X:5.3f}")
        
        # 4. Leakage summary
        total_leak = sum(Leak[i,k].X for k in J_interval)
        print(f"\n💧 LEAKAGE: {total_leak:.3f} kg")
        
else:
    print(f"❌ Optimization failed. Status: {m.status}")
    print("💡 Check constraints or increase MIPGap/time limit")





Set parameter Username
Academic license - for non-commercial use only - expires 2026-11-05
Set parameter MIPGap to value 0.001
Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (win64)

CPU model: Intel(R) Core(TM) i5-10210U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 843 rows, 700 columns and 1950 nonzeros
Model fingerprint: 0x87bf0e93
Model has 110 quadratic constraints
Variable types: 300 continuous, 400 integer (240 binary)
Coefficient statistics:
  Matrix range     [1e-02, 2e+01]
  QMatrix range    [7e-03, 1e+00]
  QLMatrix range   [1e-02, 4e+00]
  Objective range  [1e+01, 1e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e-01, 1e+02]
Presolve removed 330 rows and 350 columns
Presolve time: 0.23s
Presolved: 933 rows, 470 columns, 2880 nonzeros
Variable types: 240 continuous, 230 integer (170 binary)

Root relaxation: objective 3.646083e+02, 428 iterations, 0.01 seconds 